In [1]:
%pip install --upgrade pip
%pip install --disable-pip-version-check \
    torch==1.13.1 \
    torchdata==0.5.1 --quiet

%pip install \
    transformers==4.27.2 \
    datasets==2.11.0 \
    evaluate==0.4.0 \
    rouge_score==0.1.2 \
    loralib==0.1.1 \
    peft==0.3.0 \
    trl==0.4.4 --quiet


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    GenerationConfig
)
from datasets import load_dataset 
from peft import PeftConfig, PeftModel, LoraConfig, TaskType

# trl: transformer reinforcement learning library 
from trl import PPOConfig, PPOTrainer, AutoModelForSeq2SeqLMWithValueHead
from trl import create_reference_model 
from trl.core import LengthSampler

import torch 
import evaluate 

import numpy as np 
import pandas as pd 

# tqdm library makes the loops show a smart progress meter. 
from tqdm import tqdm 
tqdm.pandas()

In [3]:
model_name = "google/flan-t5-base" 
hf_dataset_name = "knkarthick/dialogsum"

dataset_original = load_dataset(hf_dataset_name)
dataset_original

ValueError: Invalid pattern: '**' can only be an entire path component

In [5]:
def build_dataset(model_name,
        dataset_name,
        input_min_text_length,
        input_max_text_length): 
    # load dataset 
    dataset = load_dataset(dataset_name, split="train")
    dataset = dataset.filter(
        lambda x: len(x["dialogue"]) > input_min_text_length and 
        len(x["dialogue"]) <= input_max_text_length
        )
    
    tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto")

    def tokenize(sample):
        # wrap each dialogue with the instruction 
        prompt = f""" 
Summarize the following conversation.

{sample["dialogue"]}

Summary: 
""" 
        sample["input_ids"] = tokenizer.encode(prompt)

        # this must be called "query", which is a requirement 
        sample["query"] = tokenizer.decode(sample["input_ids"])

        return sample 
    
    # tokenize each dialogue 
    dataset = dataset.map(tokenize, batched=False)
    dataset.set_format(type="torch")

    # split the dataset into train and test parts 
    datasets_splits = dataset.train_test_split(
        test_size=0.2, shuffle=False, seed=42 
    )

    return datasets_splits 

dataset = build_dataset(model_name=model_name,
                        dataset_name=hf_dataset_name,
                        input_min_text_length=200,
                        input_max_text_length=1000)

print(dataset)

ValueError: Invalid pattern: '**' can only be an entire path component